<a href="https://colab.research.google.com/github/mooxY-ttt/computational-linguistics-portfolio/blob/main/02_ner_relation_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the spaCy library
!pip install spacy

# Download the pre-trained small English model (en_core_web_sm)
# This model includes components for tokenization, POS tagging, and NER
!python -m spacy download en_core_web_sm

import spacy
import pandas as pd

# Load the pre-trained model into memory
nlp = spacy.load("en_core_web_sm")

print("Environment setup complete. Model loaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Environment setup complete. Model loaded successfully.


In [3]:
# Sample unstructured text containing multiple PERSON and ORG entities
news_text = """
Apple Inc. is reportedly in advanced talks to acquire the AI startup Windsurf.
Tim Cook, the CEO of Apple, believes this move will strengthen their position in the tech market.
Meanwhile, Elon Musk's company Tesla is facing new regulations in Europe.
Musk recently tweeted about his concerns regarding AI safety, a topic also discussed by Sam Altman at OpenAI.
"""

print(f"Corpus loaded. Total characters: {len(news_text)}")

Corpus loaded. Total characters: 365


In [4]:
# Process the text through the spaCy pipeline to create a Doc object
doc = nlp(news_text)

# Initialize lists to store extracted entities
persons = []
organizations = []

# Iterate through all recognized entities in the document
for ent in doc.ents:
    if ent.label_ == "PERSON":
        persons.append(ent.text)
    elif ent.label_ == "ORG":
        organizations.append(ent.text)

# Use set() to remove duplicate entries
print("Extracted PERSON entities:", list(set(persons)))
print("Extracted ORG entities:", list(set(organizations)))

Extracted PERSON entities: ['Tim Cook', 'Sam Altman', "Elon Musk's"]
Extracted ORG entities: ['AI', 'Apple Inc.', 'Tesla', 'Apple']


In [5]:
# Initialize a list to store the extracted relational tuples
relations = []

# Iterate through each sentence detected by spaCy's sentence segmenter
for sent in doc.sents:
    sent_persons = []
    sent_orgs = []

    # Extract entities specific to the current sentence
    for ent in sent.ents:
        if ent.label_ == "PERSON":
            sent_persons.append(ent.text)
        elif ent.label_ == "ORG":
            sent_orgs.append(ent.text)

    # If both a PERSON and an ORG exist in the same sentence, create a relation tuple
    if sent_persons and sent_orgs:
        for p in sent_persons:
            for o in sent_orgs:
                relations.append({
                    "Person": p,
                    "Organization": o,
                    "Context": sent.text.strip() # Store the original sentence as context
                })

print(f"Extraction complete. Found {len(relations)} potential relations.")

Extraction complete. Found 2 potential relations.


In [6]:
# Convert the list of dictionaries into a pandas DataFrame
df_relations = pd.DataFrame(relations)

# Display the structured output
df_relations

,Person,Organization,Context
0,Tim Cook,Apple,"Tim Cook, the CEO of Apple, believes this move..."
1,Elon Musk's,Tesla,"Meanwhile, Elon Musk's company Tesla is facing..."
